# A small data analysis with the additional modules

To illustrate the usefulness of the additional modules we are going to go through a very small analysis of DNA sequences with python.

For our task, we have been given a set of sequence IDs corresponding to the [Cytochrome c Oxidase subunit I](https://en.wikipedia.org/wiki/Cytochrome_c_oxidase_subunit_I) mitochondrial gene in various organisms belonging to 2 groups: *Metazoans* (animals), and *Viridiplantae* (plant). Our goal is to get to the sequences corresponding to these IDs, compute their GC percents, and then compare the GC percent of the plant and animal sequences.

---

We start by importing the different modules:

In [ ]:
import pandas as pd

import Bio

import matplotlib.pyplot as plt 

import scipy.stats as stats

## 1. reading the sequence ids

We use [pandas](https://pandas.pydata.org/) to read the sequences ids which are in a csv file.

Pandas is a great module when it comes to reading, manipulating, and writing tabular data.

In [ ]:
## reading a dsv file with semicolon (;) separators:
df = pd.read_csv("data/COI_sequences.csv",sep=';')
df

In [ ]:
## selecting a column
df.record_id 

In [ ]:
## counting the different elements in the taxonomy column
df.taxonomy.value_counts()

## 2. getting the sequences

For this part we use [Biopython](https://biopython.org/), which is the go-to library for biological sequence manipulation (alignment, translation, ...) as well as interaction with online databases such as Swiss-Prot or NCBI's Entrez.

---

We will first try with a single sequence:

In [ ]:
from Bio import Entrez
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

Entrez.email = "wandrille.duchemin@unibas.ch"  # write your e-mail here. Always tell NCBI who you are

## query with a sequence id
stream = Entrez.efetch(db="nucleotide", id='EU701247.1', rettype="fasta", retmode="text")

## we receive the data as a text stream in the fasta forma, which we need to interpret
result = SeqIO.read(stream,'fasta') 

stream.close()

result

From there, we could make a for loop and make one request per sequence, but that is actually quite slow.

It is better to make a single request with multiple IDs:

In [ ]:
%%time
sequences = []

## fetching all sequences 
stream = Entrez.efetch(db="nucleotide", id=list(df.record_id),
                       rettype="fasta", retmode="text")

## parsing the sequences records:
for record in SeqIO.parse(stream,'fasta'):

    seq = record.seq
    sequences.append( seq )

stream.close()

In [ ]:
print( *( sequences[:5] ) , '...' , sep='\n***\n')

## 3. compute the sequences GC fraction

While it would not be too difficult to write a function to compute the GC fraction (ie. the fraction of nucleotides which are either G or C) ourselves, we can use the one provided by Biopython directly:

In [ ]:
from Bio.SeqUtils import gc_fraction

GC = []
for seq in sequences:
    GC.append( gc_fraction(seq) )

print( *( GC[:5] ) , '...' , sep=' , ')

## 4. GC fraction numerical analysis

We add this as a new column to the table:

In [ ]:
# adding the GC fraction as a new column on the dataframe
df["GC"] = GC
df.head()

We can now leverate pandas's function to do the analysis,
in particular using the `groupby` function

In [ ]:
df.groupby('taxonomy').GC.mean()

In [ ]:
## we can separate data from the two categories
df.groupby('taxonomy').GC.agg(list)

To test if the two groups have statistically different means, we are going to use the non-parametric Mann-Whitney U test,
which can be found in python as part of the [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) module ([scipy](https://docs.scipy.org/doc/scipy/index.html) is a larger collection of modules).

In [ ]:
import scipy.stats as stats

## extracting the GCs as a dictionary where the keys are 'Metazoa' or 'Viridiplantae'
## and the values are the corresponding list of GC fractions
GCs = df.groupby('taxonomy').GC.agg(list).to_dict()

## the function returns two values: the test statistic (U), and the associated p-value
U,pv = stats.mannwhitneyu( GCs['Metazoa'],
                           GCs['Viridiplantae'] )
print( "Mann-Whitney U test p-value:" , pv)

## 5. plotting the data

In addition to the test, it is nice to represent the data we are analysing.

We can do so with [matplotlib](https://matplotlib.org/) which is the go-to library for plotting in python:

In [ ]:
import matplotlib.pyplot as plt 


_ = plt.boxplot( GCs.values() ,tick_labels = GCs.keys() )

**extra**: we can make further plots with the [seaborn](https://seaborn.pydata.org/) library, which is based on matplotlib and offers an interface that is very appropriate for the analysis of tabular data

> there is no seaborn extra notebook, because it is what comes after matplotlib, but we do propose an [intermediate python course](https://github.com/sib-swiss/intermediate-python-training) which teaches seaborn.

In [ ]:
import seaborn as sns

sns.violinplot( df,  x = 'taxonomy' , y = 'GC'  , cut = 0)
sns.stripplot( df,  x = 'taxonomy' , y = 'GC' , dodge=True, color = 'grey', alpha = 0.8)

